# check_exact: verificación exacta de la geometría generada

Para cada escalón evalúa el espesor de PLA en el centro del escalón y lo compara con el diseño (espesor del escalón k = (N-k)·c75); para las láminas compara el espesor total.

**Requiere** `check_geom.py` en la carpeta del notebook (se importa igual que en el script original).

## 1. Imports

In [ ]:
import sys, re, glob, os, importlib.util
import numpy as np

## 2. Parámetros
Carpetas de casos a verificar (en el script original eran los argumentos de la línea de comandos).

In [ ]:
args = sorted(glob.glob("casos/*"))
here = os.getcwd()          # carpeta donde esta check_geom.py
args

## 3. Importar `check_geom.py`
`check_geom.py` procesa `sys.argv[1:]` al importarse, por eso se recorta `sys.argv` (importante en Jupyter, donde `sys.argv` trae los argumentos del kernel).

In [ ]:
sys.argv = sys.argv[:1]
spec = importlib.util.spec_from_file_location("cg", os.path.join(here, "check_geom.py"))
cg = importlib.util.module_from_spec(spec); spec.loader.exec_module(cg)

## 4. Funciones y datos de referencia

In [ ]:
ORIG = [0.04, 0.04, 0.05, 0.05, 0.05, 0.06, 0.07, 0.09, 0.18]
def thick(su, x):
    surf, sets, val = cg.load(su); t = open(su).read().replace('\r', '')
    c75 = val('c75'); c77 = val('c77')
    top = val(re.search(r'^1629\s+%pin_TR%\s+PZ\s+(.*?)\s*(\$|$)', t, flags=re.M).group(1))
    zz = np.linspace(c77 + 1e-9, top - 1e-9, 4000)
    return np.mean([cg.inside(sets, surf, x, z) for z in zz]) * (top - c77), c75, val('c71')

## 5. Verificación caso por caso

In [ ]:
ok = True
for d in [x for x in args if os.path.isdir(x)]:
    name = os.path.basename(d.rstrip('/'))
    su = [s for s in glob.glob(d + '/dat/Su_ridge_filter_*.inp') if (not s.endswith('_9.inp')) or name == 'ref_N9'][0]
    desc = glob.glob(d + '/ridge_*.txt'); lam = glob.glob(d + '/lamina_*.txt')
    if lam:
        n = int(re.search(r'flat(\d+)', su).group(1)); exp = max(n, 1) * 0.02
        th1, _, c71 = thick(su, 0.3); th2, _, _ = thick(su, 0.01)
        good = abs(th1 - exp) < 1e-3 and abs(th2 - exp) < 1e-3; ok &= good
        print("%-14s lamina n=%-2d espesor %.3f cm (esperado %.3f)%s  %s" % (name, n, th1, exp, "  [material AIRE]" if n == 0 else "", "OK" if good else "ERROR"))
        continue
    if desc:
        txt = open(desc[0]).read().split('espesor 0):')[-1]
        w = [float(x) for x in re.findall(r'float64\(([0-9.]+)\)', txt)] or [float(x) for x in re.findall(r'[0-9]+\.[0-9]+|[0-9]+e-?[0-9]+', txt)]
    else:
        w = ORIG
    N = len(w); cum = 0.0; bad = []
    for k, wk in enumerate(w, 1):
        xc = cum + wk / 4            # centro del escalon k en x>0: tramo [cum, cum+wk/2]
        th, c75, c71 = thick(su, xc); cum += wk / 2
        if abs(th - (N - k) * c75) > 1e-3: bad.append((k, round(th, 3), round((N - k) * c75, 3)))
    ok &= not bad
    print("%-14s N=%-2d periodo=%.4f cm  %s %s" % (name, N, sum(w), "OK" if not bad else "ERROR", bad if bad else ""))

In [ ]:
print("TODO OK" if ok else "HAY ERRORES")